<a href="https://colab.research.google.com/github/soniaeya/COMP6321-Project/blob/main/COMP6321_Final_Project_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🌎 Global Variables

## Imports

In [184]:
import numpy as np

## Functions

### 1. Loading Datasets

In [185]:
import os
import torch

# Set Kaggle API credentials as environment variables
# Replace "YOUR_KAGGLE_USERNAME" and "YOUR_KAGGLE_KEY" with your actual credentials
os.environ['KAGGLE_USERNAME'] = "soniaeya"
os.environ['KAGGLE_KEY'] = "257652ae96445cceae0d2358d7834357"

from kaggle.api.kaggle_api_extended import KaggleApi
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_dataset(kaggle_url, dataset_num, dataset_name):

    DATASET = kaggle_url # kaggle dataset url
    BASE_ROOT = "./datasets" # where images will be downloaded on the local machine
    DATASET_NAME = DATASET.split("/")[-1]  # "dataset-3-animals"
    ROOT = os.path.join(BASE_ROOT, DATASET_NAME)

    api = KaggleApi()
    api.authenticate()

    # downloading the images from kaggle
    if not os.path.exists(ROOT) or len(os.listdir(ROOT)) == 0:
        os.makedirs(ROOT, exist_ok=True)
        api.dataset_download_files(DATASET, path=ROOT, unzip=True)

    base = os.path.join(ROOT, dataset_num , dataset_name)

    #Resize to 224x224 and convert to tensor
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize( # For tranfer learning with imagenet
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    # Creating Pytorch dataset with ImageFolder
    dataset = ImageFolder(base, transform=transform)

    # Split sizes
    total_size = len(dataset)
    train_size = int(0.8 * total_size)
    val_size = int(0.10 * total_size)
    test_size = total_size - train_size - val_size

    # Deterministic split
    generator = torch.Generator().manual_seed(42)

    train_dataset, val_dataset, test_dataset = random_split(
        dataset,
        [train_size, val_size, test_size],
        generator=generator
    )
    batch_size = 32

    # Create loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=4,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader


# Return a Dataloader of the images in Dataset 2



### 2. Creating the ResNet Skeleton

In [186]:
import torch.optim as optim
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np
import os


### 3. Creating T-SNE Graphs

In [194]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import torch # Ensure torch is imported for device

# Global helper function for t-SNE plotting
def plot_tsne_features(model_path, data_loader, plot_title, model_builder_func):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Build the model using the provided model_builder_func
    model = model_builder_func() # model_builder_func should return an untrained model
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()

    # Remove classification head to get feature embedder
    embedder = torch.nn.Sequential(*(list(model.children())[:-1]))
    embedder = embedder.to(device).eval()

    all_features = []
    all_labels = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = embedder(images)
            outputs = outputs.squeeze(-1).squeeze(-1)  # (B, 2048)

            all_features.append(outputs.cpu())
            all_labels.append(labels)

    features = torch.cat(all_features, dim=0).numpy()
    labels = torch.cat(all_labels, dim=0).numpy()

    print("Feature matrix shape:", features.shape)
    print("Labels shape:", labels.shape)

    # Perform t-SNE
    tsne_reducer = TSNE(
        n_components=2,
        perplexity=30,
        init="pca",
        learning_rate="auto",
        random_state=42
    )

    X_2d = tsne_reducer.fit_transform(features)

    # Plotting
    plt.figure()
    plt.scatter(X_2d[:, 0], X_2d[:, 1], c=labels, s=10)
    plt.colorbar()
    plt.title(plot_title)
    plt.show()


## Values

In [188]:
import os

custom_model_dataset_1_model_path = "custom_model_dataset_1.pth"
MODEL_SAVE_BASE_DIR = "/content/drive/MyDrive/COMP6321-Final-Project-Code/models"

imagenet_dataset_2_model_path = os.path.join(MODEL_SAVE_BASE_DIR, "imagenet_dataset_2.pth")
imagenet_dataset_3_model_path = os.path.join(MODEL_SAVE_BASE_DIR, "imagenet_dataset_3.pth")

# 🎨 Custom Model

##1. Custom Model x Dataset 1

### 1.1 Creating and training Custom Model on Dataset 1


### 1.1 Loading Dataset 1

In [189]:
@staticmethod
def load_dataset_1():
  torch.manual_seed(42)
  device = "cuda" if torch.cuda.is_available() else "cpu"

  # Load dataset with correct parameters
  train_loader, val_loader, test_loader = load_dataset("soniaeya/dataset-1-colorectal-cancer", "Dataset 1", "Colorectal Cancer ")

  return train_loader, val_loader, test_loader

### 1.2 Training Custom Model on Dataset 1

In [195]:
import json
import os

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
from torchvision import transforms
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

import torch
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())
class CustomModel:
  def __init__(self, loader):
        train_loader, val_loader, test_loader = loader
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

  train_metrics = []
  val_metrics = []
  # Same preprocessing as preprocessing script
  data_transforms = transforms.Compose([
      transforms.Resize((224, 224)),
      transforms.ToTensor(),
      transforms.Normalize(
          mean=[0.485, 0.456, 0.406],
          std=[0.229, 0.224, 0.225]
      )
  ])


  def build_model(self, num_classes=3):
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, int(num_classes), dtype=torch.float32)
        return model

  def train_one_epoch(self, model, dataloader, criterion, optimizer, device):
      # Switch model to training mode (enables dropout, updates batchnorm stats)
      model.train()
      running_loss = 0.0
      correct = 0
      total = 0
      all_labels = []
      all_preds = []

      for images, labels in dataloader:
          images, labels = images.to(device), labels.to(device)

          optimizer.zero_grad()
          outputs = model(images)
          loss = criterion(outputs, labels)
          loss.backward()
          optimizer.step()

          running_loss += loss.item() * images.size(0)
          preds = outputs.argmax(dim=1)
          correct += (preds == labels).sum().item()
          total += labels.size(0)

          all_labels.extend(labels.cpu().numpy())
          all_preds.extend(preds.cpu().numpy())

      epoch_loss = running_loss / total
      epoch_acc = correct / total

      precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
      recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
      f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

      return epoch_loss, epoch_acc, precision, recall, f1

  def evaluate(self, model, dataloader, criterion, device):
      # Switch model to evaluation mode (disables dropout, uses running batchnorm stats)
      model.eval()
      loss_sum = 0.0
      correct = 0
      total = 0
      all_labels = []
      all_preds = []

      with torch.no_grad():
          for images, labels in dataloader:
              images, labels = images.to(device), labels.to(device)

              outputs = model(images)
              loss = criterion(outputs, labels)

              loss_sum += loss.item() * images.size(0)
              preds = outputs.argmax(dim=1)
              correct += (preds == labels).sum().item()

              all_labels.extend(labels.cpu().numpy())
              all_preds.extend(preds.cpu().numpy())
              total += labels.size(0)

      avg_loss = loss_sum / total
      accuracy = correct / total

      precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
      recall = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
      f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

      return avg_loss, accuracy, precision, recall, f1

  def process_dataset_1(self):
      best_path = custom_model_dataset_1_model_path # Use the global variable

      # 1. Build model
      model = self.build_model()

      # 2. Loss and optimizer
      criterion = nn.CrossEntropyLoss()
      optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

      # 3. Device handling
      device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
      model = model.to(device)

      # 4. Training configuration
      num_epochs = 1

      train_losses = []
      val_losses = []
      train_accuracies = []
      val_accuracies = []



      best_val_loss = float("inf")
      os.makedirs("outputs", exist_ok=True)

          # Check if model already exists
      if os.path.exists(best_path):
          model.load_state_dict(torch.load(best_path, map_location=device))
          print(f"Model already exists at {best_path}. Loading existing model.")


      # 5. Training loop
      for epoch in range(num_epochs):

          train_loss, train_acc, train_precision, train_recall, train_f1 = self.train_one_epoch(
              model, self.train_loader, criterion, optimizer, device
          )

          val_loss, val_acc, val_precision, val_recall, val_f1 = self.evaluate(
              model, self.val_loader, criterion, device
          )

          train_losses.append(train_loss)
          val_losses.append(val_loss)
          train_accuracies.append(train_acc)
          val_accuracies.append(val_acc)

          # Save best model
          if val_loss < best_val_loss:
              best_val_loss = val_loss
              torch.save(model.state_dict(), best_path)

          print(f"Epoch {epoch+1}/{num_epochs} | \n"
                f"\nTrain Loss {train_loss:.4f} \nTrain Accuracy {train_acc:.2f}% \nTrain Precision {train_precision:.4f} \nTrain Recall {train_recall:.4f}\nTrain F1 {train_f1:.4f}\n\n"
                f"Validation Loss {val_loss:.4f} \nValidation Accuracy {val_acc:.2f}% \nValidation Precision {val_precision:.4f} \nValidation Recall {val_recall:.4f} \nValidation F1 {val_f1:.4f}")

          # Creating np array for new metric, and append to the class's metrics array
          new_train_metric = np.array([train_loss, train_acc, train_precision, train_recall, train_f1])
          self.train_metrics.append(new_train_metric)

          new_val_metric = np.array([val_loss, val_acc, val_precision, val_recall, val_f1])
          self.val_metrics.append(new_val_metric)


      model = self.build_model()
      model.load_state_dict(torch.load(best_path, map_location=device))
      model = model.to(device)
      model.eval()

      # Final evaluation on the test set
      test_loss, test_acc, test_precision, test_recall, test_f1 = self.evaluate(model, self.test_loader, criterion, device)
      print(f"\nTest Loss: {test_loss:.4f}")
      print(f"Test Accuracy:  {test_acc*100:.2f}%")
      print(f"Test Precision: {test_precision:.4f}")
      print(f"Test Recall:    {test_recall:.4f}")
      print(f"Test F1-Score:  {test_f1:.4f}\n")

      # Remove classification head
      encoder = torch.nn.Sequential(*(list(model.children())[:-1]))
      encoder = encoder.to(device)
      encoder.eval()

      all_features = []
      all_labels = []

      with torch.no_grad():
          for images, labels in self.val_loader:
              images = images.to(device)

              outputs = encoder(images)
              outputs = outputs.squeeze(-1).squeeze(-1)  # (B, 2048)

              all_features.append(outputs.cpu())
              all_labels.append(labels)

      # Concatenate all batches
      all_features = torch.cat(all_features, dim=0).numpy()
      all_labels = torch.cat(all_labels, dim=0).numpy()

      print("Feature matrix shape:", all_features.shape)
      print("Labels shape:", all_labels.shape)
  def tsne_dataset_1(self):
    # Call the global plot_tsne_features function
    plot_tsne_features(custom_model_dataset_1_model_path, self.test_loader, "Dataset 1 t-SNE Visualization", self.build_model)

True
12.8
1


In [191]:
custom_model = CustomModel(load_dataset_1())
custom_model.process_dataset_1()

Model already exists at custom_model_dataset_1.pth. Loading existing model.
Epoch 1/1 | 

Train Loss 0.0088 
Train Accuracy 1.00% 
Train Precision 0.9965 
Train Recall 0.9965
Train F1 0.9965

Validation Loss 0.1048 
Validation Accuracy 0.96% 
Validation Precision 0.9651 
Validation Recall 0.9650 
Validation F1 0.9650

Test Loss: 0.1910
Test Accuracy:  96.17%
Test Precision: 0.9617
Test Recall:    0.9617
Test F1-Score:  0.9616

Feature matrix shape: (600, 2048)
Labels shape: (600,)


In [192]:
print(custom_model.train_metrics)

[array([0.00876922, 0.99645833, 0.99645944, 0.99645833, 0.99645831])]


In [193]:
custom_model.tsne_dataset_1()

TypeError: tsne() missing 1 required positional argument: 'plot_title'

#### Graph about metrics evolution through epochs in Custom Model vs. ImageNet

##2. Custom Model x Dataset 2

##3. Custom Model x Dataset 3

# 🐻 ImageNet Model

## ImageNet x Dataset 2

### 1. Creating and training ImageNet on Dataset 2


#### Loading Dataset 2

In [ ]:
def load_dataset_2():
    torch.manual_seed(42)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load dataset
    train_loader, val_loader, test_loader = load_dataset("soniaeya/dataset-2-prostate-cancer", "Dataset 2", "Prostate Cancer")

    return train_loader, val_loader, test_loader

#### Training ImageNet on Dataset 2

In [196]:
from google.colab import drive

class ImageNet:
  def __init__(self, loader):
    train_loader, val_loader, test_loader = loader
    self.train_loader = train_loader
    self.val_loader = val_loader
    self.test_loader = test_loader
    self.train_metrics = []
    self.val_metrics = []

  @staticmethod
  def get_imagenet_model(num_classes=3, device=device):
    weights = models.ResNet50_Weights.IMAGENET1K_V2 # Define weights before its first use
    model = models.resnet50(weights=weights)

    # Replace classifier head
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    model = model.to(device)
    return model

  def train_model(self, model, train_loader, val_loader, epochs, device):

      criterion = nn.CrossEntropyLoss()
      optimizer = optim.Adam(model.parameters(), lr=1e-4)

      best_val_acc = -1.0 # Initialize with a very low value for accuracy
      best_model_state = None
      best_epoch = -1

      for epoch in range(epochs):

          # ---------------- TRAIN ----------------
          model.train()
          running_loss = 0.0
          correct = 0
          total = 0
          all_train_labels = []
          all_train_preds = []
          for images, labels in train_loader:
              images, labels = images.to(device), labels.to(device)

              optimizer.zero_grad()
              outputs = model(images)
              loss = criterion(outputs, labels)
              loss.backward()
              optimizer.step()

              running_loss += loss.item() * images.size(0)
              preds = outputs.argmax(dim=1)
              correct += (preds == labels).sum().item()
              total += labels.size(0)

              all_train_labels.extend(labels.cpu().numpy())
              all_train_preds.extend(preds.cpu().numpy())

          train_loss = running_loss / total
          train_acc = correct / total # Accuracy as decimal

          train_precision = precision_score(all_train_labels, all_train_preds, average='weighted', zero_division=0)
          train_recall = recall_score(all_train_labels, all_train_preds, average='weighted', zero_division=0)
          train_f1 = f1_score(all_train_labels, all_train_preds, average='weighted', zero_division=0)


          # ---------------- VALIDATION ----------------
          val_loss, val_acc, val_precision, val_recall, val_f1 = self.evaluate_model(model, val_loader, device)

          self.train_metrics.append(np.array([train_loss, train_acc, train_precision, train_recall, train_f1]))

          self.val_metrics.append(np.array([val_loss, val_acc, val_precision, val_recall, val_f1]))

          print(f"Epoch {epoch+1}/{epochs} | \n"
                f"Train Loss {train_loss:.4f} Train Accuracy {train_acc*100:.2f}%
"f"Train Precision {train_precision:.4f} Train Recall {train_recall:.4f} Train F1 {train_f1:.4f}\n"
                f"Val loss {val_loss:.4f} acc {val_acc*100:.2f}% precision {val_precision:.4f} recall {val_recall:.4f} f1 {val_f1:.4f}")

      return model

  def evaluate_model(self, model, test_loader, device):
      criterion = nn.CrossEntropyLoss()

      model.eval()
      loss_sum = 0.0
      correct = 0
      total = 0

      with torch.no_grad():
          for images, labels in test_loader:
              images, labels = images.to(device), labels.to(device)

              outputs = model(images)
              loss = criterion(outputs, labels)

              loss_sum += loss.item() * images.size(0)
              preds = outputs.argmax(dim=1)
              correct += (preds == labels).sum().item()
              total += labels.size(0)

      avg_loss = loss_sum / total
      accuracy = correct / total # Return accuracy as decimal
      model.eval()
      all_labels = []
      all_preds = []

      with torch.no_grad():
          for images, labels in test_loader:
              images, labels = images.to(device), labels.to(device)
              outputs = model(images)
              preds = outputs.argmax(dim=1)

              all_labels.extend(labels.cpu().numpy())
              all_preds.extend(preds.cpu().numpy())

      # Calculate metrics
      precision = precision_score(all_labels, all_preds, average='weighted')
      recall = recall_score(all_labels, all_preds, average='weighted')
      f1 = f1_score(all_labels, all_preds, average='weighted')

      print(f"\nTest Precision: {precision:.4f}")
      print(f"Test Recall:    {recall:.4f}")
      print(f"Test F1-Score:  {f1:.4f}")

      return avg_loss, accuracy, precision, recall, f1

  def process_dataset(self, model_path):
      drive.mount('/content/drive', force_remount=True) # Mount drive at the start
      torch.manual_seed(42)
      device = "cuda" if torch.cuda.is_available() else "cpu"

      # Load dataset
      train_loader, val_loader, test_loader = self.loader

      # Define model save path
      model_save_path = model_path # Using the global variable

      # Ensure model directory exists
      os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

      # Get model skeleton
      model = self.get_imagenet_model(num_classes=3, device=device)

      # Check if model already exists
      if os.path.exists(model_save_path):
          model.load_state_dict(torch.load(model_save_path, map_location=device))
          print(f"Model already exists at {model_save_path}. Loading existing model.")
      else:
          # Train the model
          print(f"No model found at {model_save_path}. Starting training...")
          model = self.train_model(
              model,
              train_loader,
              val_loader,
              epochs=5,
              device=device
          )
          torch.save(model.state_dict(), model_save_path)
          print(f"Trained model saved to {model_save_path}")

      # Test the model (either loaded or trained)
      test_loss, test_acc, test_precision, test_recall, test_f1 = self.evaluate_model(model, test_loader, device)
      print(f"\nTest Loss: {test_loss:.4f}")
      print(f"\nTest Accuracy:  {test_acc*100:.2f}%\n")


  # tsne("imagenet_dataset_3.pth", load_dataset_3(), "Dataset 3 t-SNE Visualization")
  def tsne(self, model_path, load_dataset, plot_title):
      # Call the global plot_tsne_features function
      plot_tsne_features(model_path, self.test_loader, plot_title, self.get_imagenet_model)


  @torch.no_grad()
  def extract_features(loader, embedder):
      features, labels = [], []

      for images, targets in loader:
          images = images.to(device)

          outputs = embedder(images)      # (B, 2048, 1, 1)
          outputs = outputs.squeeze(-1).squeeze(-1)  # (B, 2048)

          features.append(outputs.cpu())
          labels.append(targets)

      features = torch.cat(features).numpy()
      labels = torch.cat(labels).numpy()

      return features, labels


SyntaxError: unterminated f-string literal (detected at line 75) (1283601948.py, line 75)

### 2. Getting the test results from ImageNet on Dataset 2

In [ ]:
imagenet_model = ImageNet()
imagenet_model.process_dataset(load_dataset_2, imagenet_dataset_2_model_path)

In [ ]:
#print(pretrained_dataset_2_metrics)

### 3. T-SNE Analysis

In [ ]:
tsne(imagenet_dataset_2_model_path, load_dataset_2(), "Dataset 2 t-SNE Visualization")

## ImageNet x Dataset 3

### 1. Creating and training ImageNet on Dataset 3

#### Loading Dataset 3

In [ ]:
def load_dataset_3():
    torch.manual_seed(42)

    # Load dataset
    train_loader, val_loader, test_loader = load_dataset("soniaeya/dataset-3-animals", "Dataset 3", "Animal Faces")
    return train_loader, val_loader, test_loader

#### Training ImageNet on Dataset 3

In [ ]:
def pretrained_model_dataset_3():
    train_loader, val_loader, test_loader = load_dataset_3()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Define device here

    # Define model save path
    model_save_path = imagenet_dataset_3_model_path # Using the global variable

    # Ensure model directory exists
    os.makedirs(os.path.dirname(model_save_path), exist_ok=True)

    # Get model skeleton
    model = get_imagenet_model(num_classes=3, device=device)

    # Check if model already exists
    if os.path.exists(model_save_path):
        model.load_state_dict(torch.load(model_save_path, map_location=device))
        print(f"Model already exists at {model_save_path}. Loading existing model.")
    else:
        # Train the model
        print(f"No model found at {model_save_path}. Starting training...")
        model = train_model(
            model,
            train_loader,
            val_loader,
            epochs=5,
            device=device
        )
        torch.save(model.state_dict(), model_save_path) # Re-added this line to save the model
        print(f"Trained model saved to {model_save_path}")

    # Test the model (either loaded or trained)
    test_loss, test_acc, test_precision, test_recall, test_f1 = evaluate_model(model, test_loader, device)
    # Removed: test_acc /= 100 as evaluate_model now returns decimal accuracy
    print(f"\nTest Loss: {test_loss:.4f}")
    print(f"Test Accuracy:  {test_acc*100:.2f}%") # Print as percentage


    return test_loss, test_acc, test_precision, test_recall, test_f1

### 2. Getting the test results from ImageNet on Dataset 3

In [ ]:
pretrained_dataset_3_metrics = np.array(pretrained_model_dataset_3())

### 3. T-SNE Analysis

In [ ]:
tsne(imagenet_dataset_3_model_path, load_dataset_3(), "Dataset 3 t-SNE Visualization")

## ImageNet Results Analysis

In [ ]:
pretrained_metrics = np.vstack((pretrained_dataset_2_metrics, pretrained_dataset_3_metrics))
pretrained_metrics *= 100


In [ ]:

# Metric labels (excluding loss)
labels = ["Dataset 2", "Dataset 3"]

# Convert precision, recall, F1 to percentage

pretrained_loss, pretrained_accuracy, pretrained_precision, pretrained_recall, pretrained_f1 = pretrained_metrics[:, 0], pretrained_metrics[:, 1], pretrained_metrics[:, 2], pretrained_metrics[:, 3], pretrained_metrics[:, 4]

x = np.arange(len(labels))

plt.figure()
plt.plot(x, pretrained_loss, marker='o', label="Loss")

plt.xticks(x, labels)
# Removed plt.ylim(95, 100) as it's not appropriate for loss values
plt.ylabel("Loss Value")
plt.title("Test Loss Comparison")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Create a DataFrame for plotting
metric_names = ['Loss', 'Accuracy', 'Precision', 'Recall', 'F1-Score']
data_for_plot = []

for i, label in enumerate(labels):
    for j, metric_name in enumerate(metric_names):
        # Exclude Loss from this plot
        if metric_name == 'Loss':
            continue
        data_for_plot.append({
            'Dataset': label,
            'Metric': metric_name,
            'Value': pretrained_metrics[i, j]
        })

df_metrics = pd.DataFrame(data_for_plot)

plt.figure(figsize=(10, 6))
bars = sns.barplot(x='Metric', y='Value', hue='Dataset', data=df_metrics, palette='viridis')

# Make each bar thinner
for container in bars.containers:
    for patch in container.patches:
        patch.set_width(0.25) # Adjust this value as needed (e.g., 0.25 for thinner bars)

# Overlay line plots to connect the tops of the bars for each dataset
sns.lineplot(x='Metric', y='Value', hue='Dataset', data=df_metrics,
             marker='o', markersize=8, legend=False, linewidth=2, color='black') # Added black color and linewidth for visibility

# Add values on top of the bars
for container in plt.gca().containers:
    for patch in container.patches:
        plt.gca().annotate(f'{patch.get_height():.2f}',
                           (patch.get_x() + patch.get_width() / 2, patch.get_height()),
                           ha='center', va='bottom',
                           color='black', fontsize=9, xytext=(0, 5), textcoords='offset points')

plt.ylim(96, 100)
plt.ylabel("Percentage (%)")
plt.title("ImageNet Performance on Dataset 2 and 3")
plt.legend(title="Dataset")

# Set y-axis ticks to display with two decimal places and set major locator to 0.10
plt.gca().yaxis.set_major_locator(mticker.MultipleLocator(0.25))
plt.gca().yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))

plt.grid(axis='y', linestyle='--', alpha=0.7)

# Adjust layout to create more padding
plt.tight_layout() # Automatically adjusts subplot params for a tight layout

plt.show()